# Data Quality - Update Checks Notebook

Use this notebook to update existing rows in `check_registry` without reloading your full add-check list.

Identity key for updates: `(check_name, workspace_id, dataset_id)`.

In [ ]:
# Configuration
# Preferred: load shared settings from Fabric config notebook.
try:
    ip = get_ipython()
    if ip is None:
        raise RuntimeError("IPython runtime not available")
    ip.run_line_magic("run", "data_quality_config_notebook")
except Exception:
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
print(f"Schema: {FULL_SCHEMA}")

In [ ]:
from IPython.display import display

display(spark.sql(f"""
SELECT
    check_name, model_name, workspace_id, dataset_id,
    workspace_name, dataset_name, run_frequency, is_active, is_baseline
FROM {FULL_SCHEMA}.check_registry
ORDER BY check_name, model_name, workspace_id, dataset_id
""").toPandas())


## How To Specify Updates

Each tuple is:
`(check_name, workspace_id, dataset_id, model_name, workspace_name, dataset_name, dax_expression, run_frequency, is_active, is_baseline)`

Rules:
- First 3 fields are required identity selectors
- For update fields, use `None` to keep existing value unchanged
- `run_frequency` (when provided) must be `ONCE_PER_DAY` or `MULTIPLE_PER_DAY`
- `is_baseline = True` designates this row as baseline for its `check_name` and clears baseline from other rows for that check
- Clearing the current baseline (`is_baseline = False`) is rejected unless another row in the same submission is promoted to baseline
- If a check has exactly one active row, that row is always auto-forced to baseline


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import trim, upper
from IPython.display import display

# -- EDIT BELOW ---------------------------------------------------------------
updates = [
    # Example: rename model label, update DAX, and optionally set as baseline
    (
        "Total Sales",                                     # check_name       — required
        "66666666-7777-8888-9999-000000000000",            # workspace_id     — required
        "ffffffff-1111-2222-3333-444444444444",            # dataset_id       — required
        "Sales AMER v2",                                   # model_name       — None to keep unchanged
        None,                                              # workspace_name   — None to keep unchanged
        None,                                              # dataset_name     — None to keep unchanged
        'EVALUATE ROW("value", [Total Revenue])',          # dax_expression   — None to keep unchanged
        None,                                              # run_frequency    — None to keep unchanged
        None,                                              # is_active        — None to keep unchanged
        None,                                              # is_baseline      — True to promote baseline, False to clear (guarded), None to keep unchanged
    )
]
# -----------------------------------------------------------------------------

if len(updates) == 0:
    raise ValueError("No updates provided. Add at least one row to the updates list.")

schema = StructType([
    StructField("check_name",    StringType(),  False),
    StructField("workspace_id",  StringType(),  False),
    StructField("dataset_id",    StringType(),  False),
    StructField("model_name",    StringType(),  True),
    StructField("workspace_name",StringType(),  True),
    StructField("dataset_name",  StringType(),  True),
    StructField("dax_expression",StringType(),  True),
    StructField("run_frequency", StringType(),  True),
    StructField("is_active",     BooleanType(), True),
    StructField("is_baseline",   BooleanType(), True),
])

u = spark.createDataFrame(updates, schema=schema)
u = (
    u.withColumn("check_name",    trim("check_name"))
     .withColumn("workspace_id",  trim("workspace_id"))
     .withColumn("dataset_id",    trim("dataset_id"))
     .withColumn("model_name",    trim("model_name"))
     .withColumn("workspace_name",trim("workspace_name"))
     .withColumn("dataset_name",  trim("dataset_name"))
     .withColumn("dax_expression",trim("dax_expression"))
     .withColumn("run_frequency", upper(trim("run_frequency")))
)

bad_selector_rows = u.filter(
    "check_name IS NULL OR check_name = '' OR workspace_id IS NULL OR workspace_id = '' OR dataset_id IS NULL OR dataset_id = ''"
).toPandas()
if len(bad_selector_rows) > 0:
    raise ValueError("Update rows must include non-blank check_name/workspace_id/dataset_id.")

dupe_selectors = u.groupBy("check_name", "workspace_id", "dataset_id").count().filter("count > 1").toPandas()
if len(dupe_selectors) > 0:
    raise ValueError("Duplicate update selectors found in submission.")

bad_frequency_rows = u.filter(
    "run_frequency IS NOT NULL AND run_frequency <> '' AND run_frequency NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')"
).toPandas()
if len(bad_frequency_rows) > 0:
    raise ValueError("Invalid run_frequency in updates. Allowed: ONCE_PER_DAY, MULTIPLE_PER_DAY")

u.createOrReplaceTempView("_updates")

missing_targets = spark.sql(f"""
    SELECT u.check_name, u.workspace_id, u.dataset_id
    FROM _updates u
    LEFT JOIN {FULL_SCHEMA}.check_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
    WHERE t.check_name IS NULL
""").toPandas()
if len(missing_targets) > 0:
    raise RuntimeError("One or more update selectors do not exist in check_registry.")

# Prevent multiple promotions in one submission for the same check_name.
baseline_promotions = u.filter("is_baseline = true").groupBy("check_name").count().filter("count > 1").toPandas()
if len(baseline_promotions) > 0:
    raise ValueError(
        "Submission promotes multiple baseline rows for the same check_name. "
        "Provide at most one is_baseline=True row per check_name."
    )

# Baseline rows must stay active after update.
inactive_promotion = spark.sql(f"""
    SELECT u.check_name, u.workspace_id, u.dataset_id
    FROM _updates u
    INNER JOIN {FULL_SCHEMA}.check_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
    WHERE u.is_baseline = true
      AND COALESCE(u.is_active, t.is_active) = false
""").toPandas()
if len(inactive_promotion) > 0:
    raise ValueError("A baseline row must be active. Set is_active=True or remove is_baseline=True for these updates.")

# Reject clearing the current baseline unless the same submission promotes a replacement baseline.
clearing_without_replacement = spark.sql(f"""
    SELECT DISTINCT t.check_name
    FROM _updates u
    INNER JOIN {FULL_SCHEMA}.check_registry t
      ON u.check_name = t.check_name
     AND u.workspace_id = t.workspace_id
     AND u.dataset_id = t.dataset_id
    WHERE u.is_baseline = false
      AND t.is_baseline = true
      AND t.is_active = true
      AND NOT EXISTS (
          SELECT 1
          FROM _updates u2
          WHERE u2.check_name = t.check_name
            AND u2.is_baseline = true
      )
""").toPandas()
if len(clearing_without_replacement) > 0:
    projected_active_counts = spark.sql(f"""
        WITH projected AS (
            SELECT
                t.check_name,
                CASE
                    WHEN u.is_active IS NULL THEN t.is_active
                    ELSE u.is_active
                END AS projected_is_active
            FROM {FULL_SCHEMA}.check_registry t
            LEFT JOIN _updates u
              ON u.check_name = t.check_name
             AND u.workspace_id = t.workspace_id
             AND u.dataset_id = t.dataset_id
        )
        SELECT
            check_name,
            SUM(CASE WHEN projected_is_active = true THEN 1 ELSE 0 END) AS projected_active_count
        FROM projected
        GROUP BY check_name
    """).toPandas()

    blocked_checks = clearing_without_replacement.merge(
        projected_active_counts,
        on='check_name',
        how='left'
    )
    blocked_checks = blocked_checks[blocked_checks['projected_active_count'] > 0]
    if len(blocked_checks) > 0:
        raise ValueError(
            "Cannot clear the current baseline without assigning a new one when active rows remain. "
            "Use is_baseline=True on the replacement row in the same submission."
        )

spark.sql(f"""
    MERGE INTO {FULL_SCHEMA}.check_registry t
    USING _updates s
      ON t.check_name = s.check_name
     AND t.workspace_id = s.workspace_id
     AND t.dataset_id = s.dataset_id
    WHEN MATCHED THEN UPDATE SET
      t.model_name     = COALESCE(s.model_name,     t.model_name),
      t.workspace_name = COALESCE(s.workspace_name, t.workspace_name),
      t.dataset_name   = COALESCE(s.dataset_name,   t.dataset_name),
      t.dax_expression = COALESCE(s.dax_expression, t.dax_expression),
      t.run_frequency  = COALESCE(CASE WHEN s.run_frequency = '' THEN NULL ELSE s.run_frequency END, t.run_frequency),
      t.is_active      = COALESCE(s.is_active,      t.is_active),
      t.is_baseline    = COALESCE(s.is_baseline,    t.is_baseline)
""")

# When a row is promoted to baseline (is_baseline=True), clear the flag on all other rows for the check_name.
newly_baseline_df = spark.sql(f"""
    SELECT t.check_name, t.workspace_id, t.dataset_id
    FROM {FULL_SCHEMA}.check_registry t
    INNER JOIN _updates u
      ON t.check_name = u.check_name
     AND t.workspace_id = u.workspace_id
     AND t.dataset_id = u.dataset_id
    WHERE t.is_baseline = true
      AND t.is_active = true
      AND u.is_baseline = true
""").toPandas()

for _, row in newly_baseline_df.iterrows():
    cn  = row["check_name"].replace("'", "''")
    wid = row["workspace_id"].replace("'", "''")
    did = row["dataset_id"].replace("'", "''")
    spark.sql(f"""
        UPDATE {FULL_SCHEMA}.check_registry
        SET is_baseline = false
        WHERE check_name = '{cn}'
          AND is_active = true
          AND NOT (workspace_id = '{wid}' AND dataset_id = '{did}')
    """)
    print(f"  Transferred baseline for '{row['check_name']}' -> workspace={row['workspace_id']} / dataset={row['dataset_id']}")

# Force policy: if a check has exactly one active row, that row must be the baseline.
spark.sql(f"""
    UPDATE {FULL_SCHEMA}.check_registry
    SET is_baseline = true
    WHERE is_active = true
      AND check_name IN (
          SELECT check_name
          FROM {FULL_SCHEMA}.check_registry
          WHERE is_active = true
          GROUP BY check_name
          HAVING COUNT(*) = 1
      )
""")

# Enforce strict baseline invariant for active checks only: exactly one active baseline per active check_name.
active_baseline_issues = spark.sql(f"""
    SELECT
        check_name,
        COUNT(*) AS active_row_count,
        SUM(CASE WHEN is_baseline = true THEN 1 ELSE 0 END) AS active_baseline_count
    FROM {FULL_SCHEMA}.check_registry
    WHERE is_active = true
    GROUP BY check_name
    HAVING SUM(CASE WHEN is_baseline = true THEN 1 ELSE 0 END) <> 1
""").toPandas()
if len(active_baseline_issues) > 0:
    raise RuntimeError(
        "Baseline invariant violation after update: each active check_name must have exactly one active baseline row.\n"
        + active_baseline_issues.head(20).to_string(index=False)
    )

frequency_conflicts = spark.sql(f"""
    SELECT check_name
    FROM {FULL_SCHEMA}.check_registry
    GROUP BY check_name
    HAVING COUNT(DISTINCT UPPER(TRIM(COALESCE(run_frequency, '')))) > 1
""").toPandas()
if len(frequency_conflicts) > 0:
    raise RuntimeError(
        "run_frequency consistency violation detected after update for check_name(s): "
        + ", ".join(frequency_conflicts["check_name"].tolist())
    )

print(f"Applied updates to {len(updates)} row selector(s).")

# --- Verify updated rows ---
updated_rows = spark.sql(f"""
SELECT
    t.check_name, t.model_name, t.workspace_id, t.dataset_id,
    t.workspace_name, t.dataset_name, t.run_frequency, t.is_active, t.is_baseline
FROM {FULL_SCHEMA}.check_registry t
INNER JOIN _updates u
  ON t.check_name = u.check_name
 AND t.workspace_id = u.workspace_id
 AND t.dataset_id = u.dataset_id
ORDER BY t.check_name, t.model_name, t.workspace_id, t.dataset_id
""").toPandas()

display(updated_rows)
print(f"\nVerified {len(updated_rows)} updated row(s).")
